# Phase 6: Deployment

**CRISP-DM Phase Description:**  
The final phase involves organising and presenting the project findings so that the customer can use them, or integrating the model into a live production system. Depending on the requirements, deployment can range from generating a simple report to implementing a repeatable data mining process across the enterprise.

---

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import json

---
### Task 1: Plan Deployment

Develop a strategy for putting the data mining results to practical use. This includes:

- **Deployment Strategy:** How will the model be consumed? Options include a REST API, a batch prediction script, an interactive dashboard, or integration into an existing software system.
- **Infrastructure Requirements:** What technical environment is needed (e.g., cloud services, web server, containerisation)?
- **Steps to Deploy:** List the concrete, ordered steps needed to move from a trained model to a live, usable product.
- **User Training:** Will end-users need instructions or documentation to use the deployed solution?

**Instructions:** Outline your deployment plan as a structured dictionary. If building an API or script, sketch the basic structure in code.

In [2]:
# TODO: Plan the deployment of your model/solution.

import sys
sys.path.append('..')

deployment_plan = {
    "strategy": "Batch Prediction Tool — The trained Random Forest model is saved as a serialised joblib file and loaded by a lightweight Python script. Before each harvest season (typically 3 months in advance), a planner provides district-level Annual_Rainfall forecasts and Fertilizer usage data. The script loads the model, encodes the inputs, and returns predicted yield_per_hectare for each Rice or Wheat district.",
    "target_users": "Agricultural logistics teams, state storage planners, and food supply chain coordinators.",
    "deployment_format": "Saved sklearn Pipeline (joblib). Invoked via a CLI script or imported into a planning dashboard.",
    "input_requirements": [
        "Annual_Rainfall (mm) — forecast from weather bureau",
        "Fertilizer (kg) — from district agricultural records",
        "Pesticide (kg) — from district agricultural records",
        "State and Season — known in advance",
        "Area (hectare) — from land registry"
    ],
    "output": "Predicted yield_per_hectare for each district. Planners multiply by Area to estimate total production tonnage for storage allocation.",
    "retraining_cadence": "Annually. After each harvest season, new yield actuals are appended to the training set and the model is retrained with the same pipeline to reflect updated crop patterns.",
    "limitations": [
        "Model was trained on data from 1997-2020. Performance may degrade for extreme climate events outside this range.",
        "Zone feature gives broad geographic context only. District-level enrichment would improve predictions further.",
        "Cross-validation showed variance (R2 range 0.67-0.89). Predictions in underrepresented states should be treated with caution."
    ]
}

import json
print(json.dumps(deployment_plan, indent=2))


{
  "strategy": "Batch Prediction Tool \u2014 The trained Random Forest model is saved as a serialised joblib file and loaded by a lightweight Python script. Before each harvest season (typically 3 months in advance), a planner provides district-level Annual_Rainfall forecasts and Fertilizer usage data. The script loads the model, encodes the inputs, and returns predicted yield_per_hectare for each Rice or Wheat district.",
  "target_users": "Agricultural logistics teams, state storage planners, and food supply chain coordinators.",
  "deployment_format": "Saved sklearn Pipeline (joblib). Invoked via a CLI script or imported into a planning dashboard.",
  "input_requirements": [
    "Annual_Rainfall (mm) \u2014 forecast from weather bureau",
    "Fertilizer (kg) \u2014 from district agricultural records",
    "Pesticide (kg) \u2014 from district agricultural records",
    "State and Season \u2014 known in advance",
    "Area (hectare) \u2014 from land registry"
  ],
  "output": "Predic

In [3]:
# Save the best fitted model to disk for deployment use
import joblib, os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Re-load the post-scraping dataset and retrain for saving
df = pd.read_csv('../data/processed/crop_yield_cleaned.csv')
TARGET = 'yield_per_hectare'
X = df.drop(columns=[TARGET])
y = df[TARGET]

final_model = Pipeline([('regressor', RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42))])
final_model.fit(X, y)  # Train on full dataset for deployment

os.makedirs('../models', exist_ok=True)
MODEL_PATH = '../models/crop_yield_rf_model.joblib'
joblib.dump(final_model, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')
print(f'Input features: {list(X.columns)}')


Model saved to: ../models/crop_yield_rf_model.joblib
Input features: ['Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Crop_Wheat', 'Season_Kharif', 'Season_Rabi', 'Season_Summer', 'Season_Whole Year', 'Season_Winter', 'State_Arunachal Pradesh', 'State_Assam', 'State_Bihar', 'State_Chhattisgarh', 'State_Delhi', 'State_Goa', 'State_Gujarat', 'State_Haryana', 'State_Himachal Pradesh', 'State_Jammu and Kashmir', 'State_Jharkhand', 'State_Karnataka', 'State_Kerala', 'State_Madhya Pradesh', 'State_Maharashtra', 'State_Manipur', 'State_Meghalaya', 'State_Mizoram', 'State_Nagaland', 'State_Odisha', 'State_Puducherry', 'State_Punjab', 'State_Sikkim', 'State_Tamil Nadu', 'State_Telangana', 'State_Tripura', 'State_Uttar Pradesh', 'State_Uttarakhand', 'State_West Bengal', 'Zone_Eastern', 'Zone_North-Eastern', 'Zone_Northern', 'Zone_Southern', 'Zone_Western']


In [4]:
# Deployment plan already printed by the cell above


---
### Task 2: Plan Monitoring and Maintenance

A deployed model is not "set and forget". Over time, the data distribution may shift (**data drift**) or the model's performance may degrade (**model decay**). This task creates a monitoring and maintenance plan:

- **Performance Monitoring:** How and how often will the model's predictions be evaluated against actuals?
- **Data Drift Detection:** How will changes in the input data distribution be detected?
- **Retraining Strategy:** Under what conditions will the model be retrained (e.g., scheduled, triggered by performance drop)?
- **Logging and Alerting:** What will be logged, and what triggers an alert?

**Instructions:** Document your monitoring and maintenance plan.

In [5]:
# TODO: Plan monitoring and maintenance.

monitoring_plan = {
    "performance_monitoring": {
        "method": "After each harvest season, compare model predictions against actual yield records. Calculate MAE and R2 drift.",
        "threshold": "If R2 drops below 0.75 or MAE increases by more than 20% relative to the original baseline, trigger a retraining cycle.",
        "tooling": "Simple pandas comparison script run annually against updated crop records."
    },
    "data_drift": {
        "method": "Monitor the distribution of Annual_Rainfall inputs versus the training distribution. Extreme climate anomalies (e.g., unprecedented drought) would push inputs outside the model's learned range.",
        "action": "Flag predictions as low-confidence if input Annual_Rainfall is more than 2 standard deviations outside the training mean."
    },
    "retraining_trigger": [
        "Annual review cycle regardless of drift.",
        "Performance degradation (R2 < 0.75) detected on new season data.",
        "Major policy or agricultural practice change that alters the feature distribution."
    ],
    "future_improvements": [
        "Add district-level weather data for finer-grained predictions beyond state averages.",
        "Include multi-year rainfall trend features rather than single-year snapshots.",
        "Test Gradient Boosting (XGBoost) for potential R2 improvement.",
        "At 10TB+ scale: migrate from Pandas/sklearn to PySpark MLlib with distributed feature engineering."
    ]
}

print(json.dumps(monitoring_plan, indent=2))


{
  "performance_monitoring": {
    "method": "After each harvest season, compare model predictions against actual yield records. Calculate MAE and R2 drift.",
    "threshold": "If R2 drops below 0.75 or MAE increases by more than 20% relative to the original baseline, trigger a retraining cycle.",
    "tooling": "Simple pandas comparison script run annually against updated crop records."
  },
  "data_drift": {
    "method": "Monitor the distribution of Annual_Rainfall inputs versus the training distribution. Extreme climate anomalies (e.g., unprecedented drought) would push inputs outside the model's learned range.",
    "action": "Flag predictions as low-confidence if input Annual_Rainfall is more than 2 standard deviations outside the training mean."
  },
  "retraining_trigger": [
    "Annual review cycle regardless of drift.",
    "Performance degradation (R2 < 0.75) detected on new season data.",
    "Major policy or agricultural practice change that alters the feature distributio

In [6]:
# Monitoring plan already printed above


---
### Task 3: Produce Final Report

Compile a final summary of the entire project. This report should communicate the findings to stakeholders who may not have a technical background. It typically includes:

- **Executive Summary:** A brief overview of the problem, approach, and key results.
- **Key Findings:** The most important patterns, insights, and model performance results.
- **Visualisations:** Charts and plots that support the findings (suitable for a presentation).
- **Recommendations:** Actionable recommendations based on the analysis.
- **Limitations:** Known limitations and caveats of the analysis.

**Instructions:** Draft the final report content below. Include any code needed to generate summary visualisations.

In [7]:
# TODO: Draft the final report as a structured document.

final_report = {
    "title": "Crop Yield Prediction using CRISP-DM — KH5004CMD Data Science",
    "business_question": "Can we forecast the expected yield (Production per Hectare) of Rice and Wheat 3 months before harvest using Annual Rainfall and Fertilizer data to optimize storage and logistics planning?",
    "research_question": "Which crop (Rice or Wheat) shows the highest resilience — lowest yield variance — in years with Deficient Rainfall, and should policy encourage its cultivation in drought-prone districts?",
    "dataset": "Kaggle Indian Agriculture Dataset (1997-2020), 19,689 rows, filtered to 1,742 Rice and Wheat records.",
    "scraping": "Wikipedia States and Union Territories of India — geographic Zone classification scraped using BeautifulSoup. Merged on State key. Adds broad regional context to each record.",
    "key_findings": [
        "Random Forest (Tuned) achieved R2=0.90 on the unseen test set, well above the 0.75 success criterion.",
        "Annual_Rainfall, Fertilizer, and State were the dominant yield predictors per feature importance analysis.",
        "In deficient rainfall years (below 996mm), Rice shows Std Dev=0.93 vs. Wheat at 1.40 — Rice is the more drought-resilient crop.",
        "Adding the Wikipedia Zone feature improved R2 by only +0.0008, confirming the primary dataset already captured most geographic signal via the State dummies.",
        "Cross-validation mean R2=0.79 (range 0.67-0.89) indicates the model generalises well but has less certainty in underrepresented districts."
    ],
    "big_data_discussion": "The current 19,689 row dataset is comfortably handled by Pandas on a local machine. At 10TB+ scale this would require distributed compute. With Apache Spark and PySpark MLlib, the feature engineering and model training pipeline could be parallelised across a cluster, with HDFS or S3 as the data layer. Pandas would be replaced by Spark DataFrames. The sklearn Pipeline would need to be rewritten using Spark MLlib Pipeline stages. Dask is a lighter intermediate option that preserves the Pandas API while enabling multi-core parallelism for datasets up to single-node memory limits.",
    "ethics_note": "Some Indian states are over-represented in the dataset, which may bias yield predictions towards historically well-documented regions. Policy recommendations from the resilience analysis should be reviewed by domain experts before being applied to specific drought-prone districts."
}

# Summary print
print('=== FINAL REPORT SUMMARY ===')
print(f"Title     : {final_report['title']}")
print(f"BQ Answer : {final_report['key_findings'][0]}")
print(f"RQ Answer : {final_report['key_findings'][2]}")
print(f"Big Data  : {final_report['big_data_discussion'][:120]}...")


=== FINAL REPORT SUMMARY ===
Title     : Crop Yield Prediction using CRISP-DM — KH5004CMD Data Science
BQ Answer : Random Forest (Tuned) achieved R2=0.90 on the unseen test set, well above the 0.75 success criterion.
RQ Answer : In deficient rainfall years (below 996mm), Rice shows Std Dev=0.93 vs. Wheat at 1.40 — Rice is the more drought-resilient crop.
Big Data  : The current 19,689 row dataset is comfortably handled by Pandas on a local machine. At 10TB+ scale this would require di...


In [8]:
# Final report summary printed above


In [9]:
# Optional: Generate summary visualisations for the stakeholder presentation

# import matplotlib.pyplot as plt

# Example: Bar chart comparing model performance
# models = ['Model A', 'Model B', 'Model C']
# scores = [0.82, 0.88, 0.85]
#
# plt.figure(figsize=(8, 5))
# plt.bar(models, scores, color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='black')
# plt.title('Model Performance Comparison', fontsize=14)
# plt.ylabel('F1-Score')
# plt.ylim(0, 1)
# for i, v in enumerate(scores):
#     plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
# plt.tight_layout()
# plt.show()

---
### Task 4: Review Project

Conduct a project retrospective. Reflect on what went well, what did not go as planned, and what lessons can be carried forward to future projects.

- **What went well?** Successes, effective techniques, good decisions.
- **What could be improved?** Challenges, mistakes, or inefficiencies encountered.
- **Lessons learned:** Key takeaways for future data science projects.
- **Skills developed:** New skills or knowledge gained during the project.

**Instructions:** Complete the project retrospective below.

In [10]:
# TODO: Conduct the project retrospective.

project_retrospective = {
    "went_well": [
        "CRISP-DM lifecycle followed end-to-end across six explicit phase notebooks.",
        "Reusable src/ scripts for cleaning, feature engineering, scraping, merging, and modeling successfully decoupled logic from notebooks.",
        "Random Forest achieved R2=0.90, exceeding the stated success criterion.",
        "Research Question 2 (drought resilience) produced a clear, actionable finding: Rice is more resilient than Wheat under deficient rainfall.",
        "Before/after scraping comparison was completed honestly — the Zone feature had minimal but documented impact."
    ],
    "challenges": [
        "The raw dataset column was named 'Annual_Rainfall', not 'Rainfall' or 'Temperature' as originally assumed. This required correcting the cleaning pipeline.",
        "Wikipedia Zone merge initially produced 20.6% unmatched rows due to state name formatting differences. Resolved by adding str.strip() normalisation before the merge.",
        "Cross-validation showed meaningful variance between folds (0.67 to 0.89), indicating the model is less certain for underrepresented states."
    ],
    "if_more_time": [
        "Incorporate district-level drought severity scores as additional features.",
        "Experiment with Gradient Boosting and XGBoost for potential R2 gains.",
        "Build a minimal Streamlit dashboard for logistics planners to input district data and receive yield predictions interactively.",
        "Demonstrate Spark/Dask pipeline at scale to fully satisfy the Big Data tool comparison requirement."
    ]
}

print('\n=== PROJECT RETROSPECTIVE ===')
print('What went well:')
for item in project_retrospective['went_well']:
    print(f'  + {item}')
print('\nChallenges:')
for item in project_retrospective['challenges']:
    print(f'  - {item}')
print('\nIf more time:')
for item in project_retrospective['if_more_time']:
    print(f'  > {item}')



=== PROJECT RETROSPECTIVE ===
What went well:
  + CRISP-DM lifecycle followed end-to-end across six explicit phase notebooks.
  + Reusable src/ scripts for cleaning, feature engineering, scraping, merging, and modeling successfully decoupled logic from notebooks.
  + Random Forest achieved R2=0.90, exceeding the stated success criterion.
  + Research Question 2 (drought resilience) produced a clear, actionable finding: Rice is more resilient than Wheat under deficient rainfall.
  + Before/after scraping comparison was completed honestly — the Zone feature had minimal but documented impact.

Challenges:
  - The raw dataset column was named 'Annual_Rainfall', not 'Rainfall' or 'Temperature' as originally assumed. This required correcting the cleaning pipeline.
  - Wikipedia Zone merge initially produced 20.6% unmatched rows due to state name formatting differences. Resolved by adding str.strip() normalisation before the merge.
  - Cross-validation showed meaningful variance between fold

In [11]:
# Display the retrospective
# print("=" * 60)
# print("PROJECT RETROSPECTIVE")
# print("=" * 60)
#
# sections = [
#     ("What Went Well", project_retrospective['went_well']),
#     ("What Could Be Improved", project_retrospective['could_improve']),
#     ("Lessons Learned", project_retrospective['lessons_learned']),
#     ("Skills Developed", project_retrospective['skills_developed']),
# ]
#
# for section_name, items in sections:
#     print(f"\n--- {section_name} ---")
#     if items:
#         for item in items:
#             print(f"  - {item}")
#     else:
#         print("  [Not yet documented]")

---

## Key Takeaway on Iteration

While the six CRISP-DM phases are presented **sequentially** in these notebooks, in practice the process is **highly iterative**. Real-world data science projects rarely follow a straight line from Phase 1 to Phase 6.

Common iteration patterns include:

- **Modelling → Data Preparation:** If the model performs poorly, you may return to Phase 3 to engineer new features, clean data differently, or acquire more data.
- **Evaluation → Modelling:** If the model does not meet the business success criteria, you may go back to Phase 4 to try different algorithms or tune hyperparameters.
- **Deployment → Evaluation:** If the deployed model underperforms in production (data drift), you may return to Phase 5 to re-evaluate and then to Phase 3/4 to retrain.
- **Any Phase → Business Understanding:** New findings in later phases may redefine the business objectives or success criteria originally set in Phase 1.

```
┌───────────────────────────────────────────────────────────┐
│                     CRISP-DM Lifecycle                    │
│                                                           │
│   Phase 1 ──► Phase 2 ──► Phase 3 ──► Phase 4            │
│   Business    Data        Data        Modelling           │
│   Under.      Under.      Prep.           │               │
│     ▲                       ▲              │               │
│     │                       │              ▼               │
│     │                       └──────── Phase 5             │
│     │                                Evaluation           │
│     │                                    │                │
│     │                                    ▼                │
│     └──────────────────────────────  Phase 6              │
│                                     Deployment            │
└───────────────────────────────────────────────────────────┘
```

> **Remember:** Iteration is not failure — it is the *expected* workflow. Each cycle through the process deepens your understanding of both the data and the business problem, ultimately leading to a better solution.